#Grok models

* llama-3.3-70b-versatile        ← best for tool calling
* llama-3.1-8b-instant           ← fast, lightweight
* llama3-70b-8192
* gemma2-9b-it
* mixtral-8x7b-32768

In [23]:
currency_api='3f3c11c5b652bf8a1a695a2c'
base_currency="EUR"
target_currency="INR"
url = f"https://v6.exchangerate-api.com/v6/{currency_api}/pair/{base_currency}/{target_currency}"
url

'https://v6.exchangerate-api.com/v6/3f3c11c5b652bf8a1a695a2c/pair/EUR/INR'

In [ ]:
from langchain_community.tools import tool , BaseTool, DuckDuckGoSearchResults
from langchain_core.tools import InjectedToolArg
from pydantic import BaseModel, Field
from typing import Type, Annotated
import requests, os, io 

from langchain_groq import ChatGroq
from dotenv import load_dotenv

from langchain_core.prompts import *
from langchain_core.messages import *
from langchain_openai import ChatOpenAI



load_dotenv()

llm_cv = ChatGroq(model="llama-3.3-70b-versatile")




In [25]:
search_tool=DuckDuckGoSearchResults()
search_tool.invoke("What is the history of USD and INR conversion rate")

"snippet: View US Dollar to Indian Rupee exchange rate history for 2026-4-20 as well as high, low and average USD/INR exchange rates for 2026., title: US Dollar (USD) to Indian Rupee (INR) Exchange Rates for April 20..., link: https://www.exchange-rates.org/exchange-rate-history/usd-inr-2026-04-20, snippet: Convert 1 USD to INR with the Wise Currency Converter. Analyze historical currency charts or live US dollar / Indian rupee rates and get free rate alerts directly to your email., title: 1 US dollar to Indian rupees Exchange Rate. Convert USD/INR - Wise, link: https://wise.com/gb/currency-converter/usd-to-inr-rate?amount=1, snippet: Use Xe's currency exchange rates table to compare previous currency rates, making it easy to analyze historical exchange rates across 130+ currencies.USD – US Dollar., title: Currency Exchange Rates Table - Historical Exchange Rates | Xe, link: https://www.xe.com/currencytables/, snippet: Historical data of the 1 US Dollar to the Indian Rupee exchange rat

In [26]:
#with @ method tool 

@tool 

def get_conversion_factor(base_currency: str, target_currency: str, api_key: str = currency_api) -> float:

  """
  This function fetch the currency conversion factor b/w a given base currency and target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/{currency_api}/pair/{base_currency}/{target_currency}"

  response=requests.get(url)

  return response.json()
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1777680001,
 'time_last_update_utc': 'Sat, 02 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1777766401,
 'time_next_update_utc': 'Sun, 03 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.0415}

In [27]:
# Currency convertor factor getting tool

#pydantic class
class InputStructurePydanticClass(BaseModel):
  base_currency: str = Field(..., description="Currency to be converted from")
  target_currency: str = Field(..., description="Currency to be converted to")

#base tool class
class BaseToolClass(BaseTool):
  name: str = "get_conversion_factor"
  description: str = "This function fetch the currency conversion factor b/w a given base currency and target currency"

  args_schema: type[BaseModel] = InputStructurePydanticClass

  def _run(self, base_currency: str, target_currency: str) -> float:

      url = f"https://v6.exchangerate-api.com/v6/{currency_api}/pair/{base_currency}/{target_currency}"

      response=requests.get(url)

      return response.json()
  

get_conversion_factor_tool= BaseToolClass()

get_conversion_factor_tool.invoke({"base_currency": "USD", "target_currency": "INR"})


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1777680001,
 'time_last_update_utc': 'Sat, 02 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1777766401,
 'time_next_update_utc': 'Sun, 03 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.0415}

In [28]:
# Multiplication tool 

@tool 

def MultiplicationTool(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """This function multiply base currency with conversion rate"""
  return base_currency_value * conversion_rate


MultiplicationTool.invoke({"base_currency_value":10, "conversion_rate":95.0415})


950.415

In [29]:
# Bind tool with LLM

llm_with_tool = llm_cv.bind_tools([MultiplicationTool, get_conversion_factor_tool, search_tool])

In [38]:
# tool calling 



  
messages_list= [HumanMessage("""What is the conversion factor between UDS and INR, and base on that can you convert 10 USD to INR, also give a brief history of USD and INR conversion rate keeping in mind the """)]

ai_messages = llm_with_tool.invoke(messages_list)

ai_messages

messages_list.append(ai_messages)



In [39]:
tool_call_list=ai_messages.tool_calls
tool_call_list

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_pjHmAsKoQth4539j3tAceC7V',
  'type': 'tool_call'},
 {'name': 'MultiplicationTool',
  'args': {'base_currency_value': 10},
  'id': 'call_UcXGvIUXbs6QCL4H3ctsaPUW',
  'type': 'tool_call'}]

In [32]:
MultiplicationTool.name

'MultiplicationTool'

In [33]:
import json

for tool_call in tool_call_list:
  #execute the first tool and get the value of conversion rate
  if tool_call['name'] =="get_conversion_factor":
    tool_message1=get_conversion_factor.invoke(tool_call)
    conversion_rate=json.loads(tool_message1.content)['conversion_rate']
    messages_list.append(tool_message1)
    

  if tool_call['name']=="MultiplicationTool":
    tool_call['args']['conversion_rate']=conversion_rate
    tool_message2 = MultiplicationTool.invoke(tool_call)
    messages_list.append(tool_message2)
    

    




  # Then run the second tool with conversion rate



In [34]:
messages_list

[HumanMessage(content='What is the conversion factor between UDS and INR, and base on that can you convert 10 USD to INR, also give a brief history of USD and INR conversion rate keeping in mind the current conversion factor do some web search and get the ans for the brief history', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 214, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Db8Dop5FNuyAHCjUvtxoNy8vZA3Ic', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de9af-de64-78d2-8036-b39fd22d665a-0', tool_calls=[{'name': 'get_conver

In [37]:
print(llm_with_tool.invoke(messages_list).content)

In [36]:

currency_api=os.getenv["CURRENCY_API"]
currency_api

TypeError: 'function' object is not subscriptable